# Proyecto Guiado: Analítica de una plataforma de chat con IA

**Bootcamp Data & IA — Módulo 2 (SQL)**

---

## Contexto

Es tu primera semana como **junior data analyst** en *NovaChat*, una startup que ofrece
una plataforma de conversación con varios modelos de IA (GPT, Claude, Llama, Gemini…).
Los usuarios se registran en planes **free**, **pro** o **team**, abren conversaciones
y mandan mensajes. Cada respuesta del modelo puede recibir feedback positivo o negativo.

Tu manager te ha dado acceso de **lectura** a la base de datos de producción y te ha
pasado una lista de preguntas que necesita responder para la reunión de producto del
viernes. Tu trabajo durante esta sesión es:

1. Conectarte a la base de datos.
2. Explorar el esquema (qué tablas hay, qué columnas tienen, cómo se relacionan).
3. Resolver una a una las preguntas del manager usando SQL desde Python.
4. Dejar el notebook documentado y subirlo al repositorio del bootcamp en GitHub.

---

## Objetivos de aprendizaje

- Practicar el flujo real de un *data analyst*: pregunta de negocio → SQL → DataFrame → insight.
- Reutilizar las **funciones helper** que ya conoces (`run_query`, `execute`, `show_query`)
  pero ahora contra una base de datos **MySQL remota** (TiDB Cloud).
- Combinar `WHERE`, `GROUP BY`, `HAVING` y `JOIN` para responder preguntas concretas.
- Producir un entregable de calidad para tu portfolio.

---

## Esquema de la base de datos

| Tabla | Qué guarda |
|---|---|
| `usuarios` | usuarios registrados (id, nombre, email, país, plan, fecha_registro) |
| `modelos` | catálogo de modelos LLM (nombre, proveedor, coste por 1k tokens, fecha de lanzamiento) |
| `conversaciones` | conversaciones abiertas por un usuario contra un modelo |
| `mensajes` | mensajes individuales (rol user/assistant + tokens consumidos) |
| `feedback` | valoraciones positivas/negativas que los usuarios dan a respuestas concretas |
| `suscripciones` | histórico de suscripciones de pago (plan, fechas, importe, estado) |

## 1. Setup: instalación, imports, conexión y helpers

### Instalar dependencias

Solo es necesario la primera vez (o si abres el notebook en un entorno nuevo).
Si ya las tienes instaladas, la celda no hace nada y se ejecuta en segundos.

In [ ]:
%pip install pandas

### Imports

Antes de empezar a consultar nada, preparamos el entorno:

1. Importamos las librerías que usaremos durante toda la sesión.
2. Configuramos la conexión a la base de datos remota (TiDB Cloud, compatible con MySQL).
3. Definimos las funciones *helper* `run_query`, `execute` y `show_query` que usábamos
   en clases anteriores, **adaptadas** ahora a `mysql.connector` en lugar de `sqlite3`.

In [ ]:
# Pandas para manejar resultados como DataFrames
import pandas as pd

# SQLite para base de datos local
import sqlite3

# MySQL connector para TiDB Cloud (opcional)
try:
    import mysql.connector
except ImportError:
    mysql = None
    print("mysql-connector-python no instalado. La conexión a TiDB Cloud no estará disponible.")
    print("Instálalo con: %pip install mysql-connector-python")

# display() para imprimir DataFrames con formato bonito en notebooks
from IPython.display import display

# Configuracion de pandas para visualizar mejor los DataFrames
pd.set_option("display.max_columns", None)   # mostrar todas las columnas
pd.set_option("display.max_rows", 50)         # hasta 50 filas
pd.set_option("display.max_colwidth", 80)     # ancho maximo de columna

### Conexión a SQLite local

SQLite es una base de datos local que se almacena en un solo archivo.
No necesita servidor, usuario ni contraseña, lo que la hace ideal para
desarrollo, prototipado y análisis offline.

La base de datos `data/novachat.db` ya está creada con datos de ejemplo.
Solo necesitamos la ruta al archivo para conectarnos.

In [ ]:
# Configuracion de conexion a SQLite local.
DB_PATH = "../data/novachat.db"

# Abrimos la conexion. Si el archivo no existe, sqlite3 lo crea.
conn = sqlite3.connect(DB_PATH)

# Para que pandas pueda usar las columnas con sus nombres reales
conn.row_factory = sqlite3.Row

print(f"Conectado a base de datos local: {DB_PATH}")

### Conexión a TiDB Cloud Serverless

TiDB Cloud Serverless es un servicio gratuito compatible con el protocolo MySQL.
Para conectarnos necesitamos host, puerto, usuario, password y nombre de BBDD.

Importante: **TiDB exige conexión cifrada (SSL/TLS)**, por eso pasamos
`ssl_disabled: False` en la configuración.

> ⚠️ **Antes de subir a GitHub**: sustituye los valores reales de host, user
> y password por placeholders (por ejemplo `TU_HOST`, `TU_USUARIO`, `TU_PASSWORD`).
> **Nunca subas credenciales reales a un repo público.**

In [ ]:
# Configuracion de conexion a TiDB Cloud.
mysql_config = {
    "host":     "[URL_SERVER]",
    "port":     "[PUERTO]",
    "user":     "[USUARIO]",
    "password": "[CLAVE]",
    "database": "[BD]",
    "ssl_disabled": False,        # TiDB exige SSL/TLS
    "connect_timeout": 30,
}

# Abrimos la conexion. Si los datos son correctos, devuelve un objeto Connection.
# Si fallan las credenciales o la red, lanzara una excepcion clara aqui.
if mysql is None:
    raise ImportError("Ejecuta primero la celda de Imports para instalar mysql-connector-python")
conn = mysql.connector.connect(**mysql_config)

print(f"Conectado a {mysql_config['host']}:{mysql_config['port']}")
print(f"Base de datos: {mysql_config['database']}")

### Helpers: `run_query`, `execute` y `show_query`

Estas funciones son las mismas que vimos en *SQL desde Python* pero **adaptadas a SQLite**.
La principal diferencia es que `pandas` acepta directamente la conexión SQLite para
`read_sql_query`. Para sentencias que modifican datos usamos `commit()` para confirmar.

In [ ]:
def run_query(sql: str, params=None) -> pd.DataFrame:
    """
    Ejecuta una consulta SELECT y devuelve el resultado como DataFrame.

    Parameters
    ----------
    sql : str
        Consulta SQL (tipicamente un SELECT).
    params : tuple | list | None
        Valores a sustituir de forma segura en la consulta.

    Returns
    -------
    pd.DataFrame
        Las filas resultado, con los nombres de columna originales.
    """
    return pd.read_sql_query(sql, conn, params=params)


def execute(sql: str, params=None) -> int:
    """
    Ejecuta una sentencia que MODIFICA datos: INSERT / UPDATE / DELETE / DDL.
    """
    cur = conn.cursor()
    cur.execute(sql, params or ())
    conn.commit()
    affected = cur.rowcount
    cur.close()
    return affected


def show_query(sql: str, params=None) -> pd.DataFrame:
    """
    Igual que run_query pero ademas IMPRIME la SQL antes de ejecutarla.
    """
    print(sql.strip())
    return run_query(sql, params)

## 2. Explora la base de datos

Antes de tirar SQL "a ciegas" hay que **conocer el terreno**: qué tablas existen,
qué columnas tienen y de qué tipo son. Esto es lo primero que hace cualquier data
analyst al llegar a un proyecto nuevo.

### 2.1. Listar las tablas

En SQLite las tablas se listan consultando `sqlite_master`.

In [ ]:
# En SQLite, la lista de tablas se obtiene de sqlite_master
show_query("SELECT name AS tabla FROM sqlite_master WHERE type='table' ORDER BY name;")

### 2.2. Ver la estructura de cada tabla

`PRAGMA table_info(<tabla>)` nos da las columnas, sus tipos, si admiten NULL,
valor por defecto y claves.

In [ ]:
# Recorremos las tablas y mostramos la estructura de cada una.
# En SQLite usamos PRAGMA table_info() en lugar de DESCRIBE.
tablas = run_query("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")

for _, row in tablas.iterrows():
    nombre_tabla = row['name']
    print(f"\n--- {nombre_tabla} ---")
    # PRAGMA devuelve columnas: cid, name, type, notnull, dflt_value, pk
    display(run_query(f"PRAGMA table_info({nombre_tabla});"))

### 2.3. Echar un vistazo a los primeros registros

Antes de las preguntas del manager, miramos 3-5 filas de cada tabla para
entender qué pinta tienen los datos reales.

In [ ]:
# Vistazo rapido: primeras 5 filas de cada tabla
for tabla in ["usuarios", "modelos", "conversaciones", "mensajes", "feedback", "suscripciones"]:
    print(f"\n=== {tabla.upper()} ===")
    display(run_query(f"SELECT * FROM {tabla} LIMIT 5;"))

---

## 3. Bloque 1 — Warm-up: conoce a tus usuarios y a los modelos

> *"Antes de que entres en el detalle, dame una foto general:
> cuántos usuarios tenemos y de qué planes, qué modelos ofrecemos
> y cuáles son los más nuevos."* — el Manager

### Ejercicio 1 — Foto general de usuarios por plan

**Encargo del manager:**
> ¿Cuántos usuarios tenemos en total y cómo se reparten entre los planes
> *free*, *pro* y *team*?

**Pista:** un solo `GROUP BY` sobre la tabla `usuarios`. Si quieres, añade
también el total general en la misma consulta usando `WITH ROLLUP` (opcional).

In [ ]:
# Tu solucion aqui
show_query("""
SELECT 
    plan,
    COUNT(*) AS total_usuarios,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM usuarios), 2) AS porcentaje
FROM usuarios
GROUP BY plan
ORDER BY total_usuarios DESC;
""")


**Cómo lo presentarías al manager:** *(2-3 frases — qué dato sale, qué significa, y si procede una recomendación accionable)*

### Ejercicio 2 — Catálogo de modelos ordenado por coste

**Encargo del manager:**
> Dame el listado de modelos disponibles ordenado por coste, del más caro al más barato.
> Necesito el nombre, el proveedor y el coste por 1.000 tokens.

In [ ]:
# Tu solucion aqui
show_query("""
SELECT nombre, proveedor, coste_por_1k_tokens
FROM modelos
ORDER BY coste_por_1k_tokens DESC;
""")


**Cómo lo presentarías al manager:** *(2-3 frases — qué dato sale, qué significa, y si procede una recomendación accionable)*

### Ejercicio 3 — Modelos lanzados en 2024

**Encargo del manager:**
> Quiero ver solo los modelos que se lanzaron durante 2024,
> ordenados por fecha de lanzamiento (los más recientes primero).

**Pista:** la función `strftime('%Y', fecha)` extrae el año de una fecha en SQLite.
Otra alternativa es filtrar por rango con `BETWEEN '2024-01-01' AND '2024-12-31'`.

In [ ]:
# Tu solucion aqui
show_query("""
SELECT nombre, proveedor, coste_por_1k_tokens, fecha_lanzamiento
FROM modelos
WHERE strftime('%%Y', fecha_lanzamiento) = '2024'
ORDER BY fecha_lanzamiento DESC;
""")


In [ ]:
# (Opcional) reescribe la consulta anterior usando BETWEEN en lugar de YEAR()
show_query("""
SELECT nombre, proveedor, coste_por_1k_tokens, fecha_lanzamiento
FROM modelos
WHERE fecha_lanzamiento BETWEEN '2024-01-01' AND '2024-12-31'
ORDER BY fecha_lanzamiento DESC;
""")


**Cómo lo presentarías al manager:** *(2-3 frases — qué dato sale, qué significa, y si procede una recomendación accionable)*

---

## 4. Bloque 2 — Distribución y filtros

> *"Necesito entender de dónde nos viene el tráfico y cómo de
> activos están los usuarios en los últimos meses."* — el Manager

### Ejercicio 4 — Distribución de usuarios por país

**Encargo del manager:**
> Saca la distribución de usuarios por país, ordenada por número de usuarios.
> Solo me interesan los países con al menos 2 usuarios — el resto agrúpalos en
> "Otros" o simplemente no los muestres (lo que prefieras).

In [ ]:
# Tu solucion aqui
show_query("""
SELECT pais, COUNT(*) AS total_usuarios
FROM usuarios
WHERE pais IS NOT NULL
GROUP BY pais
HAVING total_usuarios >= 2
ORDER BY total_usuarios DESC;
""")


**Cómo lo presentarías al manager:** *(2-3 frases — qué dato sale, qué significa, y si procede una recomendación accionable)*

### Ejercicio 5 — Conversaciones por mes

**Encargo del manager:**
> Quiero un histograma del número de conversaciones por mes.
> Quédate solo con los meses en los que ha habido más de 5 conversaciones.

**Pista:** `DATE_FORMAT(fecha, '%Y-%m')` te da el año-mes como string,
ideal para agrupar.

In [ ]:
# Tu solucion aqui
show_query("""
SELECT strftime('%%Y-%%m', fecha_inicio) AS mes,
       COUNT(*) AS total_conversaciones
FROM conversaciones
GROUP BY mes
HAVING total_conversaciones > 5
ORDER BY mes;
""")


**Cómo lo presentarías al manager:** *(2-3 frases — qué dato sale, qué significa, y si procede una recomendación accionable)*

---

## 5. Bloque 3 — Métricas de uso por modelo

> *"Necesito entender qué modelos están consumiendo más tokens y cuánto
> nos están costando aproximadamente."* — el Manager

### Ejercicio 6 — Tokens totales y promedio por modelo

**Encargo del manager:**
> Para cada modelo, dame los tokens totales consumidos (input + output)
> y los tokens promedio por mensaje. Ordénalo por consumo total.

**Pista:** el coste se calcula sobre `tokens_input + tokens_output`.
Necesitarás un `JOIN` entre `mensajes`, `conversaciones` y `modelos`.

In [ ]:
# Tu solucion aqui
show_query("""
SELECT m.nombre AS modelo,
       SUM(ms.tokens_input + ms.tokens_output) AS tokens_totales,
       ROUND(AVG(ms.tokens_input + ms.tokens_output), 2) AS tokens_promedio
FROM mensajes ms
JOIN conversaciones c ON ms.conversacion_id = c.conversacion_id
JOIN modelos m ON c.modelo_id = m.modelo_id
GROUP BY m.modelo_id
ORDER BY tokens_totales DESC;
""")


**Cómo lo presentarías al manager:** *(2-3 frases — qué dato sale, qué significa, y si procede una recomendación accionable)*

### Ejercicio 7 — Modelos más populares (por número de conversaciones)

**Encargo del manager:**
> ¿Cuáles son los 5 modelos más populares medidos en número de conversaciones únicas?
> Indica también cuántos usuarios *distintos* los han usado.

**Pista:** `COUNT(DISTINCT ...)` cuenta valores únicos.

In [ ]:
# Tu solucion aqui
show_query("""
SELECT m.nombre AS modelo,
       COUNT(DISTINCT c.conversacion_id) AS conversaciones,
       COUNT(DISTINCT c.user_id) AS usuarios_distintos
FROM conversaciones c
JOIN modelos m ON c.modelo_id = m.modelo_id
GROUP BY m.modelo_id
ORDER BY conversaciones DESC
LIMIT 5;
""")


**Cómo lo presentarías al manager:** *(2-3 frases — qué dato sale, qué significa, y si procede una recomendación accionable)*

### Ejercicio 8 — Coste estimado por modelo

**Encargo del manager:**
> Necesito una estimación de **cuánto nos está costando** cada modelo en USD.
> La fórmula es: `(tokens_totales / 1000) * coste_por_1k_tokens`.
> Ordénalo del más caro al más barato.

In [ ]:
# Tu solucion aqui
show_query("""
SELECT m.nombre AS modelo,
       SUM(ms.tokens_input + ms.tokens_output) AS tokens_totales,
       ROUND(SUM(ms.tokens_input + ms.tokens_output) / 1000.0 * m.coste_por_1k_tokens, 4) AS coste_estimado_usd
FROM mensajes ms
JOIN conversaciones c ON ms.conversacion_id = c.conversacion_id
JOIN modelos m ON c.modelo_id = m.modelo_id
GROUP BY m.modelo_id
ORDER BY coste_estimado_usd DESC;
""")


**Insight de negocio:** observa cómo un modelo con relativamente pocos tokens
puede costar mucho más que otro con muchos tokens si su precio por 1k es
mucho mayor. Esto justifica por qué *NovaChat* podría querer empujar a usuarios
intensivos hacia modelos más eficientes.

**Cómo lo presentarías al manager:** *(2-3 frases — qué dato sale, qué significa, y si procede una recomendación accionable)*

---

## 6. Bloque 4 — Calidad: feedback y oportunidades de upgrade

> *"Hemos visto picos de feedback negativo. Necesito saber qué modelos tienen
> peor calidad percibida. Y de paso identifica candidatos a upgrade."* — el Manager

### Ejercicio 9 — Modelos con peor tasa de feedback

**Encargo del manager:**
> Para cada modelo, dame el % de feedbacks negativos sobre el total.
> Quédate solo con modelos que tengan al menos 5 feedbacks (para que el % sea
> estadísticamente relevante) y ordénalos por porcentaje negativo descendente.

**Patrón nuevo: `SUM(CASE WHEN ... THEN 1 ELSE 0 END)`**

Necesitamos contar **solo los feedbacks negativos** dentro de cada grupo, sin
filtrar el resto (porque el total también lo necesitamos para calcular el %).
La forma estándar en SQL es:

```sql
SUM(CASE WHEN condicion THEN 1 ELSE 0 END)
```

**Cómo se lee:** *"para cada fila del grupo, si cumple la condición pon un 1
y si no pon un 0; luego suma todos los unos y ceros"*. Como sumar unos
equivale a contarlos, esto es **un conteo condicional**.

Es el mismo concepto que un `if/else` de Python aplicado fila a fila:

```python
suma = sum(1 if fila.tipo == 'negativo' else 0 for fila in grupo)
```

En MySQL/TiDB existe un atajo: `SUM(tipo = 'negativo')` funciona igual porque
las comparaciones devuelven 1 (verdadero) o 0 (falso). Pero la forma con
`CASE WHEN` es más explícita y funciona en todos los motores SQL — es la
que conviene aprender primero.

In [ ]:
# Tu solucion aqui
show_query("""
SELECT m.nombre AS modelo,
       COUNT(f.feedback_id) AS total_feedback,
       SUM(CASE WHEN f.tipo = 'negativo' THEN 1 ELSE 0 END) AS negativos,
       ROUND(SUM(CASE WHEN f.tipo = 'negativo' THEN 1 ELSE 0 END) * 100.0 / COUNT(f.feedback_id), 2) AS pct_negativo
FROM feedback f
JOIN mensajes ms ON f.mensaje_id = ms.mensaje_id
JOIN conversaciones c ON ms.conversacion_id = c.conversacion_id
JOIN modelos m ON c.modelo_id = m.modelo_id
GROUP BY m.modelo_id
HAVING total_feedback >= 5
ORDER BY pct_negativo DESC;
""")


**Por qué importa el `HAVING total_feedback >= 5`:** sin ese filtro, un modelo
con 1 solo feedback negativo aparecería como "100% negativo", lo cual no es
representativo. Es un patrón muy común en analítica: filtrar grupos pequeños
antes de mostrar tasas o porcentajes.

**Cómo lo presentarías al manager:** *(2-3 frases — qué dato sale, qué significa, y si procede una recomendación accionable)*

### Ejercicio 10 — Candidatos a upgrade (free heavy users)

**Encargo del manager:**
> Identifica usuarios del plan **free** que han hecho **más de 4 conversaciones**.
> Son candidatos a una campaña de upgrade. Dame nombre, país, número de
> conversaciones y total de tokens consumidos, ordenado por consumo total.

**Cuidado con los `JOIN` y los conteos:** al unir `conversaciones` con `mensajes`,
una misma conversación aparece tantas veces como mensajes tenga. Si haces
`COUNT(c.conversacion_id)` cuentas filas, no conversaciones únicas. La forma
correcta es `COUNT(DISTINCT c.conversacion_id)`. La suma de tokens, en cambio,
es correcta porque cada mensaje aparece una sola vez en el JOIN.

In [ ]:
# Tu solucion aqui
show_query("""
SELECT u.nombre, u.pais,
       COUNT(DISTINCT c.conversacion_id) AS num_conversaciones,
       SUM(ms.tokens_input + ms.tokens_output) AS total_tokens
FROM usuarios u
JOIN conversaciones c ON u.user_id = c.user_id
JOIN mensajes ms ON c.conversacion_id = ms.conversacion_id
WHERE u.plan = 'free'
GROUP BY u.user_id
HAVING num_conversaciones > 4
ORDER BY total_tokens DESC;
""")


**Insight:** estos usuarios free están consumiendo cantidades de tokens propias
de un usuario pro. Pasarlos a una llamada del equipo de growth puede dar buen
ROI. Es exactamente el tipo de output que un junior data analyst entrega
en su día a día.

**Cómo lo presentarías al manager:** *(2-3 frases — qué dato sale, qué significa, y si procede una recomendación accionable)*

---

## 7. Bloque 5 — JOINs y análisis combinado

> *"Para terminar, dame dos cosas: un ranking de los usuarios que más
> nos cuestan y los modelos preferidos por cada plan."* — el Manager

### Ejercicio 11 — Top 5 usuarios por gasto en tokens (multi-tabla)

**Encargo del manager:**
> Ranking de los **5 usuarios que más gasto en tokens nos generan**, mostrando
> su nombre, plan, total de tokens y coste estimado en USD (multiplicando
> por el coste de cada modelo que han usado).

Aquí ya combinamos 4 tablas: `usuarios`, `conversaciones`, `mensajes`, `modelos`.

In [ ]:
# Tu solucion aqui
show_query("""
SELECT u.nombre, u.plan,
       SUM(ms.tokens_input + ms.tokens_output) AS total_tokens,
       ROUND(SUM((ms.tokens_input + ms.tokens_output) / 1000.0 * m.coste_por_1k_tokens), 4) AS coste_estimado_usd
FROM usuarios u
JOIN conversaciones c ON u.user_id = c.user_id
JOIN mensajes ms ON c.conversacion_id = ms.conversacion_id
JOIN modelos m ON c.modelo_id = m.modelo_id
GROUP BY u.user_id
ORDER BY coste_estimado_usd DESC
LIMIT 5;
""")


**Cómo lo presentarías al manager:** *(2-3 frases — qué dato sale, qué significa, y si procede una recomendación accionable)*

### Mini-tutorial: subconsultas

Una **subconsulta** es una consulta que va metida dentro de otra. Sirve para
"calcular un valor sobre la marcha" y usarlo en la consulta exterior — por
ejemplo, comparar contra el promedio, contra un máximo, o contra una lista
filtrada de IDs.

La forma más común y útil para empezar es la **subconsulta escalar**: una
subconsulta que devuelve **un único valor** y se compara con `=`, `>`, `<`, etc.

```sql
SELECT nombre, precio
FROM productos
WHERE precio > (SELECT AVG(precio) FROM productos);
--                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
--                    Subconsulta: devuelve UN número (el precio medio).
--                    Se ejecuta primero; luego la consulta de fuera la usa
--                    como si tuvieras escrito el valor a mano.
```

**Cómo leerla mentalmente:** "dame los productos cuyo precio es mayor que
*(lo que devuelva esta subconsulta)*". El motor calcula la subconsulta una
sola vez y luego hace el `WHERE` normal con ese número.

**Otros tipos de subconsultas** (más avanzadas, fuera del alcance de hoy):

- **Subconsulta como tabla derivada**: va dentro de un `FROM (...) AS sub` y se
  trata como si fuera una tabla más para hacer JOINs sobre ella.
- **Subconsulta correlacionada**: se ejecuta una vez por cada fila de la
  consulta exterior y referencia columnas de fuera. Potente pero compleja.

Para esta clase nos basta con la escalar.

### Ejercicio 12 — Modelos por encima del coste medio

**Encargo del manager:**
> ¿Qué modelos están **por encima del coste medio** del catálogo?
> Necesito el nombre, el proveedor y el coste por 1k tokens, ordenado del
> más caro al más barato. Son los modelos que tenemos que vigilar para
> optimización de gasto.

In [ ]:
# Tu solucion aqui
show_query("""
SELECT nombre, proveedor, coste_por_1k_tokens
FROM modelos
WHERE coste_por_1k_tokens > (SELECT AVG(coste_por_1k_tokens) FROM modelos)
ORDER BY coste_por_1k_tokens DESC;
""")


**Reto avanzado (opcional):** ¿cuál es el **modelo más usado por cada plan**
(`free`, `pro`, `team`)? Pista: necesitas una subconsulta que calcule el máximo
de conversaciones por plan, y comparar contra ese máximo en un `HAVING`. La
solución usa una subconsulta más compleja (correlacionada) y es un buen punto
para que practiquéis y expandáis vuestro conocimiento por vuestra cuenta.

**Cómo lo presentarías al manager:** *(2-3 frases — qué dato sale, qué significa, y si procede una recomendación accionable)*

### Ejercicio 13 (bonus) — Adopción de modelos lanzados en 2024

**Encargo del manager:**
> ¿Qué proporción del total de conversaciones se hace con modelos que se
> lanzaron en 2024 vs. modelos más antiguos?

Este ejercicio combina filtrado, agregación y un cálculo de proporción.

In [ ]:
# Tu solucion aqui
show_query("""
SELECT
    CASE WHEN strftime('%%Y', m.fecha_lanzamiento) >= '2024' THEN '2024+' ELSE 'Pre-2024' END AS grupo,
    COUNT(DISTINCT c.conversacion_id) AS conversaciones,
    ROUND(COUNT(DISTINCT c.conversacion_id) * 100.0 / (SELECT COUNT(*) FROM conversaciones), 2) AS porcentaje
FROM conversaciones c
JOIN modelos m ON c.modelo_id = m.modelo_id
GROUP BY grupo;
""")


**Cómo lo presentarías al manager:** *(2-3 frases — qué dato sale, qué significa, y si procede una recomendación accionable)*

---

## 8. Cierre y entregable

### Recapitulando

Hoy has practicado el flujo real de un junior data analyst:

1. **Conectarse** a una base de datos remota desde Python.
2. **Explorar** el esquema antes de tocar nada.
3. **Traducir** preguntas de negocio a SQL.
4. **Combinar** WHERE, GROUP BY, HAVING y JOINs para responder preguntas concretas.
5. **Comunicar** insights, no solo tirar queries.

### Cierre de la conexión

Es buena práctica cerrar la conexión cuando terminas:

In [ ]:
conn.close()
print("Conexion cerrada.")

### Entregable para GitHub

Este notebook (junto con el archivo `data/novachat.db`) es tu entregable. Antes de
hacer push:

- Asegúrate de que `data/novachat.db` existe en la raíz del proyecto.
- Añade un breve comentario al final con un *insight* propio:
  algo que hayas notado en los datos y no estaba en los enunciados.

> Recuerda: tu portfolio en GitHub es un argumento de venta. Un README claro
> y un notebook ejecutado limpio valen más que diez proyectos a medias.

## Insight propio

Analizando los datos, destaca un hallazgo que no estaba en los enunciados:
**GPT-3.5 Turbo es el modelo más usado ( empata en conversaciones con GPT-4o) y
el que más tokens consume, pero también tiene la peor tasa de feedback negativo
(57%).** En contraste, GPT-4o — del mismo proveedor y solo 3.3x más caro —
tiene 0% de feedback negativo y un coste total estimado similar ($0.047 vs $0.021).

Esto sugiere que NovaChat podría mejorar la satisfacción de usuario migrando
a los usuarios intensivos de GPT-3.5 Turbo hacia GPT-4o sin disparar los costes,
ya que GPT-4o es más eficiente por token y la calidad percibida es muy superior.
Sería una campaña de *upgrade silencioso* con alto ROI potencial.